# The Real-time AI Support Swarm System

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph_swarm import create_swarm, create_handoff_tool
from langgraph.checkpoint.memory import InMemorySaver

# -------------------------
# 1️⃣ LLM
# -------------------------
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# -------------------------
# 2️⃣ Billing Agent
# -------------------------
billing_agent = create_agent(
    llm,
    tools=[
        create_handoff_tool(agent_name="Router", description="Return control to router")
    ],
    system_prompt="""
    You are a Billing Support Agent.
    Handle invoices, refunds, pricing, subscriptions.
    Respond like a real SaaS support executive.
    """,
    name="Billing"
)

# -------------------------
# 3️⃣ Tech Support Agent
# -------------------------
tech_agent = create_agent(
    llm,
    tools=[
        create_handoff_tool(agent_name="Router", description="Return control to router")
    ],
    system_prompt="""
    You are a Technical Support Agent.
    Handle API errors, auth issues, bugs.
    Give step-by-step troubleshooting.
    """,
    name="Tech"
)

# -------------------------
# 4️⃣ Account Agent
# -------------------------
account_agent = create_agent(
    llm,
    tools=[
        create_handoff_tool(agent_name="Router", description="Return control to router")
    ],
    system_prompt="""
    You are an Account Support Agent.
    Handle login issues, access problems, suspensions.
    Ask clarifying questions when needed.
    """,
    name="Account"
)

# -------------------------
# 5️⃣ Router Agent (Brain)
# -------------------------
router_agent = create_agent(
    llm,
    tools=[
        create_handoff_tool(agent_name="Billing", description="Billing or pricing issue"),
        create_handoff_tool(agent_name="Tech", description="Technical or API issue"),
        create_handoff_tool(agent_name="Account", description="Login or account issue"),
    ],
    system_prompt="""
    You are an AI Support Router.
    Your ONLY job:
    - Understand the user's issue
    - Delegate to the correct agent
    - NEVER answer directly
    """,
    name="Router"
)

# -------------------------
# 6️⃣ Create Swarm
# -------------------------
workflow = create_swarm(
    agents=[router_agent, billing_agent, tech_agent, account_agent],
    default_active_agent="Router"
)

# -------------------------
# 7️⃣ Compile with memory
# -------------------------
app = workflow.compile(
    checkpointer=InMemorySaver()
)

print("✅ Real-time AI Support Swarm Initialized")

✅ Real-time AI Support Swarm Initialized


In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent
from langgraph_swarm import create_handoff_tool, create_swarm
import os

# -------------------------
# OpenAI API Key
# -------------------------
os.environ["OPENAI_API_KEY"] = "Add your API Key"
# -------------------------
# Initialize OpenAI model
# -------------------------
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# -------------------------
# Tools
# -------------------------
def question_answering(question: str):
    """Answers general knowledge questions."""
    return llm.invoke(question).content

def science_agent(question: str):
    """Answers science-related questions."""
    return llm.invoke(question).content

def extract_language_and_text(request: str):
    request = request.lower()
    to_patterns = [" to ", " in ", " into "]
    target_language = None
    text = request

    for pattern in to_patterns:
        if pattern in request:
            parts = request.split(pattern)
            if len(parts) == 2:
                text = parts[0]
                target_language = parts[1].strip()
                break

    text = text.replace("translate", "").replace("say", "").strip()
    return text.strip(), target_language

def translate_text(request: str):
    """Translates text to a specified language."""
    if any(x in request.lower() for x in ["help with", "can you", "need translation", "translate something"]):
        return (
            "I can help you translate. Use:\n"
            "'translate [text] to [language]' or '[text] in [language]'"
        )

    text, target_language = extract_language_and_text(request)

    if not target_language:
        return f"I need the target language to translate: '{text}'. Please specify the language in your next message."

    prompt = f"Translate the following text to {target_language}:\n{text}"
    return llm.invoke(prompt).content

# -------------------------
# Agents
# -------------------------
question_answering_agent = create_react_agent(
    model=llm,
    tools=[
        question_answering,
        create_handoff_tool(agent_name="science_agent"),
        create_handoff_tool(agent_name="translator_agent"),
    ],
    name="question_answering_agent",
    prompt=(
        "You are a general question answering agent. "
        "Route science questions to science_agent and "
        "translation requests to translator_agent. "
        "Only answer general knowledge questions."
    ),
)

science_agent = create_react_agent(
    model=llm,
    tools=[
        science_agent,
        create_handoff_tool(agent_name="question_answering_agent"),
        create_handoff_tool(agent_name="translator_agent"),
    ],
    name="science_agent",
    prompt=(
        "You are a science expert. Answer science questions only. "
        "Route non-science to question_answering_agent and "
        "translations to translator_agent."
    ),
)

translator_agent = create_react_agent(
    model=llm,
    tools=[
        translate_text,
        create_handoff_tool(agent_name="question_answering_agent"),
        create_handoff_tool(agent_name="science_agent"),
    ],
    name="translator_agent",
    prompt=(
        "You are a translation agent. Your main goal is to accurately translate text. "
        "When the user asks for a translation, your ONLY task is to call the 'translate_text' tool. "
        "For the 'request' argument of the 'translate_text' tool, you MUST provide the ENTIRE, UNMODIFIED content of the current human message. "
        "Do NOT parse or shorten the user's message before passing it to the 'translate_text' tool's 'request' argument. "
        "If the 'translate_text' tool returns a message indicating it needs a target language, "
        "you MUST communicate that exact message clearly to the user as your final response, "
        "without further action or modification. "
        "Route science questions to science_agent and general questions to question_answering_agent."
    ),
)

# -------------------------
# Swarm & Memory
# -------------------------
checkpoint = InMemorySaver()

workflow = create_swarm(
    agents=[question_answering_agent, science_agent, translator_agent],
    default_active_agent="question_answering_agent",
)

app = workflow.compile(checkpointer=checkpoint)

# -------------------------
# Save graph
# -------------------------
image = app.get_graph().draw_mermaid_png()
with open("swarm.png", "wb") as f:
    f.write(image)

# -------------------------
# Run
# -------------------------
config = {"configurable": {"thread_id": "1"}}

while True:
    user_input = input("\nEnter your request (or 'exit'): ")

    if user_input.lower() == "exit":
        print("Goodbye!")
        break

    result = app.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config,
    )

    for m in result["messages"]:
        print(m.pretty_print())

/tmp/ipython-input-3767763731.py:67: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  question_answering_agent = create_react_agent(
/tmp/ipython-input-3767763731.py:83: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  science_agent = create_react_agent(
/tmp/ipython-input-3767763731.py:98: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  translator_agent = create_react_agent(


================================ Human Message =================================

what is the current things happening in iran and USA
None
================================== Ai Message ==================================
Name: question_answering_agent
Tool Calls:
  question_answering (call_plhYkwtb99TOBZlcE4cTpYoc)
 Call ID: call_plhYkwtb99TOBZlcE4cTpYoc
  Args:
    question: What are the current events happening in Iran and the USA?
None
================================= Tool Message =================================
Name: question_answering

I'm unable to provide real-time updates or current events as my knowledge was last updated in October 2021. For the latest news on events in Iran and the USA, I recommend checking reliable news sources or websites for the most current information. If you have specific topics or events in mind, I can provide background information or context based on my training data.
None
================================== Ai Message =============================